In [ ]:
import numpy as np
from DDE_s import *
from Parameters_s import *
import sympy
from jitcdde import y, t, jitcdde
from concurrent.futures import ProcessPoolExecutor
import os

import warnings
warnings.filterwarnings("ignore")

_ = np.random.seed(0)                                                                                                                                                                                                                                                                                                                            

In [ ]:
all_control_parameters_sym = list(sympy_params.values()) + list(sympy_delays.values())
current_control_parameter_values = [params[k] for k in sympy_params.keys()] + [T_delay[k] for k in sympy_delays.keys()]
param_to_index = {sym: i for i, sym in enumerate(all_control_parameters_sym)}
max_delay_value = max(T_delay.values())

In [ ]:
duration=100
times = np.linspace(0, N_t, N_t * sample_num_factor+1)

In [ ]:
def sample_prior(n):
    return {
        'beta': np.random.uniform(0.6, 1.2, n),
        'phi_s': np.random.uniform(0.2, 0.6, n),
        'g_rate': np.random.uniform(0.005, 0.01, n),
    }

In [ ]:
param_names = ["beta", "phi_s", "g_rate"]

In [ ]:
def sampleResult(yy):
    samp_index = [int(i) for i in np.linspace(0, N_t * sample_num_factor, N_t+1)]
    sampled_yy = []
    for t_i in samp_index:
        sampled_yy.append(yy[t_i])
    return sampled_yy

def simulate(parameters):
    beta, phi_s, g_rate = parameters
    
    current_control_parameter_values[param_to_index[sympy_params['beta']]]=beta
    current_control_parameter_values[param_to_index[sympy_params['beta_prime']]]=beta*1.2
    current_control_parameter_values[param_to_index[sympy_params['phi_s_1']]]=phi_s
    current_control_parameter_values[param_to_index[sympy_params['g_beta']]]=g_rate

    DDE = jitcdde(f, control_pars=all_control_parameters_sym, max_delay=max_delay_value)
    DDE.set_integration_parameters(atol=1e-6, rtol=1e-3)
    DDE.constant_past(init_cond)
    DDE.set_parameters(current_control_parameter_values)

    data = [DDE.integrate(time) for time in times]
    yy = sampleResult(data)
    
    return yy

def simulator(sim_par):
    beta = sim_par[0]
    phi_s = sim_par[1]
    g_rate = sim_par[2]

    yy = simulate(parameters=[beta, phi_s, g_rate])
    
    pos = []

    indices = np.arange(0, N_t * sample_num_factor, sample_num_factor)
    
    pos = [(yy[t][states['F_AT_1']] + yy[t][states['F_ST_1']]) * (params['True_P_1']) +
           (yy[t][states['F_AT_2']] + yy[t][states['F_ST_2']]) * (params['True_P_2']) +
           (1 - params["True_N_1"]) * (yy[t][states['F_NT_1']] + yy[t][states['F_AT_3']] +
                                       yy[t][states['F_FT_1']] + yy[t][states['F_GT1']] +
                                       yy[t][states['F_ST_4']] + yy[t][states['F_ST_3']] +
                                       yy[t][states['F_FT_3']]) +
           (1 - params["True_N_2"]) * (yy[t][states['F_NT_2']] + yy[t][states['F_AT_4']] +
                                       yy[t][states['F_FT_2']] + yy[t][states['F_GT2']]) for t in range(N_t)]

    
    neg = [(yy[t][states['F_AT_1']] + yy[t][states['F_ST_1']]) * (1 - params['True_P_1']) +
           (yy[t][states['F_AT_2']] + yy[t][states['F_ST_2']]) * (1 - params['True_P_2']) +
           params["True_N_1"] * (yy[t][states['F_NT_1']] + yy[t][states['F_AT_3']] +
                                 yy[t][states['F_FT_1']] + yy[t][states['F_GT1']] +
                                 yy[t][states['F_ST_4']] + yy[t][states['F_ST_3']] +
                                 yy[t][states['F_FT_3']]) +
           params["True_N_2"] * (yy[t][states['F_NT_2']] + yy[t][states['F_AT_4']] +
                                 yy[t][states['F_FT_2']] + yy[t][states['F_GT2']]) for t in range(N_t)]


    hospital = [sum(yy[t][states['F_H1']] + yy[t][states['F_H2']] + yy[t][states['F_H3']] for t in range(ti + 1)) for ti
                in range(N_t)]

    death = [yy[t][states['D']] for t in range(N_t)]

    return np.stack([pos, neg, hospital, death],axis=1)

In [ ]:
def poisson_noise(simulation):
    
    with_noise = np.random.poisson(np.maximum(simulation, 1e-6))

    return with_noise

In [ ]:
import torch
from torch.distributions import Uniform, Exponential, Cauchy
from sbi.utils import MultipleIndependent

In [ ]:
beta_prior  = Uniform(torch.tensor([0.6]), torch.tensor([1.2]))
phi_s_prior = Uniform(torch.tensor([0.2]), torch.tensor([0.6]))
g_rate_prior = Uniform(torch.tensor([0.005]), torch.tensor([0.01]))

In [ ]:
prior = MultipleIndependent(
    [
        beta_prior,
        phi_s_prior,
        g_rate_prior,
    ],
    validate_args=False,
)

In [ ]:
def _sim_chunk(thetas_chunk):
    out = [simulator(th) for th in thetas_chunk]
    return out  

def simulate_many(theta, n_workers=None, chunk_size=32):

    theta = np.asarray(theta)
    n_workers = n_workers or os.cpu_count() or 4

    chunks = [theta[i:i+chunk_size] for i in range(0, len(theta), chunk_size)]
    results = [None]*len(chunks)


    with ProcessPoolExecutor(max_workers=n_workers) as ex:
        futures = {ex.submit(_sim_chunk, ch): idx for idx, ch in enumerate(chunks)}
        for fut in futures:
            idx = futures[fut]
            results[idx] = fut.result()  

    out_list = [arr for chunk in results for arr in chunk]   
    return np.stack(out_list, axis=0)                         

In [ ]:
sample_num = 10_000
thetas = prior.sample((sample_num,))
xs = simulate_many(thetas, n_workers=128, chunk_size=32)